## Breakpoint Analysis (Whole Corpus)

### Aim
Our earlier correlations and descriptive trends were computed on the **full 10-K text corpus**. To avoid testing a different object (e.g., Risk Factors only), the breakpoint hypothesis must be evaluated on a **firm–year corpus-level ESG signal** aggregated across all available sections.

### Hypotheses
- **H1 (Paris/2015 break):** ESG disclosure intensity shifts upward starting **2015/2016**.
- **H2 (2020–2021 break):** the dominant regime shift occurs around **2020–2021**.

### Corpus-Level Panel Construction
Build a `firm_year_corpus` dataset (one row per firm-year) containing:
- `E_hits`, `S_hits`, `G_hits` (lexicon term match counts across the entire corpus)
- `tokens` (total token count across all sections)
- `E_rate`, `S_rate`, `G_rate` = hits per 1,000 tokens (length-normalised intensity)
- `n_sections` (how many distinct sections are present per firm-year, for coverage / robustness)
- identifiers: `ticker`, `cik`, `year`, `gics_sector`

This corpus approach reduces section-specific bias and aligns the breakpoint test with the scope used in the correlation stage.

### Core Tests
- Breakpoint regressions comparing candidate breaks (2015/16 vs 2020/21) using the same dependent variables (`E_rate`, `S_rate`, `G_rate`).
- Piecewise trend models to compare break magnitudes and model fit (AIC/BIC).

### Robustness
- Placebo breaks (2017–2019) to show the effect is not generic time drift.
- Sector interactions to test heterogeneity across GICS sectors.
- Term ablation (remove most common terms and re-run) to reduce boilerplate sensitivity.

### Deliverables
- Plot: mean `E_rate` over time with break lines at 2015 and 2021 (with uncertainty bands).
- Table: break coefficients + AIC/BIC comparisons for each candidate breakpoint.
- Placebo plot: estimated “break effect” by candidate year.


In [13]:
# Imports
from pathlib import Path
import polars as pl
import yaml

# --------------------
# Output helpers (tables -> SQLite)
# --------------------
OUT_DIR = Path("outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

DB_PATH = OUT_DIR / "esg_panel.sqlite"
DB_URI = f"sqlite:///{DB_PATH}"  # SQLAlchemy-style URI Polars expects

def save_table_sqlite(df: pl.DataFrame, table_name: str) -> Path:
    # Overwrite table each run
    df.write_database(
        table_name=table_name,
        connection=DB_URI,
        if_table_exists="replace",
    )
    print(f"Saved table → {DB_PATH} (table: {table_name})")
    return DB_PATH

# Optional: create a couple of useful indexes (SQLite setup)
def sqlite_setup_indexes() -> None:
    import sqlite3
    con = sqlite3.connect(DB_PATH)
    cur = con.cursor()

    # Fast lookups / joins
    cur.execute("CREATE INDEX IF NOT EXISTS idx_fyc_ticker_year ON firm_year_corpus(ticker, year);")
    cur.execute("CREATE INDEX IF NOT EXISTS idx_fyc_cik_year ON firm_year_corpus(cik, year);")
    cur.execute("CREATE INDEX IF NOT EXISTS idx_fyc_sector_year ON firm_year_corpus(gics_sector, year);")

    con.commit()
    con.close()
    print("SQLite setup → indexes created/verified")

# --------------------
# Config
# --------------------
PARQUET_FILE = "spy_10k_2015_present.parquet"
LEXICON_FILE = "ESG_Lexicon.yml"

# --------------------
# Load data
# --------------------
df = pl.read_parquet(PARQUET_FILE)

# Basic hygiene
df = df.with_columns(
    pl.col("cik").cast(pl.Utf8),              # Stable key across ticker changes
    pl.col("ticker").cast(pl.Utf8),
    pl.col("company_name").cast(pl.Utf8),
    pl.col("section").cast(pl.Utf8),
    pl.col("text").cast(pl.Utf8),
    pl.col("gics_sector").cast(pl.Utf8),
    pl.col("filing_date").cast(pl.Date),      # Needed for year index
)

# Required columns check
required = {"ticker", "cik", "filing_date", "gics_sector", "section", "text"}
missing = required - set(df.columns)
assert not missing, f"Missing columns: {missing}"

# Year key
df = df.with_columns(
    pl.col("filing_date").dt.year().alias("year")  # Filing year convention
)

# Token count (count runs of non-whitespace)
df = df.with_columns(
    pl.col("text")
      .fill_null("")
      .str.count_matches(r"\S+")
      .alias("tokens")
)

# --------------------
# Load lexicon
# --------------------
with open(LEXICON_FILE, "r", encoding="utf-8") as f:
    lex = yaml.safe_load(f)

E_terms = list(lex.get("environmental") or [])
S_terms = list(lex.get("social") or [])
G_terms = list(lex.get("governance") or [])
assert E_terms and S_terms and G_terms, "Lexicon pillars are empty or missing keys"

def pillar_hits_expr(terms: list[str], text_col: str = "text") -> pl.Expr:
    # Sum regex match counts across all patterns in a pillar
    return pl.sum_horizontal([pl.col(text_col).str.count_matches(t) for t in terms])

# Add section-level hit counts
df = df.with_columns(
    pillar_hits_expr(E_terms).alias("E_hits"),
    pillar_hits_expr(S_terms).alias("S_hits"),
    pillar_hits_expr(G_terms).alias("G_hits"),
)

# --------------------
# Aggregate to firm-year corpus (all sections)
# --------------------
firm_year_corpus = (
    df.group_by(["ticker", "cik", "year", "gics_sector"])
      .agg(
          pl.sum("tokens").alias("tokens"),
          pl.sum("E_hits").alias("E_hits"),
          pl.sum("S_hits").alias("S_hits"),
          pl.sum("G_hits").alias("G_hits"),
          pl.n_unique("section").alias("n_sections"),
      )
      .filter(pl.col("tokens") > 0)  # Avoid divide-by-zero
      .with_columns(
          (pl.col("E_hits") * 1000 / pl.col("tokens")).alias("E_rate"),
          (pl.col("S_hits") * 1000 / pl.col("tokens")).alias("S_rate"),
          (pl.col("G_hits") * 1000 / pl.col("tokens")).alias("G_rate"),
      )
)

# --------------------
# Save outputs (workspace-first: SQLite DB)
# --------------------
save_table_sqlite(firm_year_corpus, "firm_year_corpus")
sqlite_setup_indexes()

# Lightweight console preview (no notebook rendering dependency)
print(
    firm_year_corpus.select([
        "ticker","cik","year","gics_sector","tokens",
        "E_hits","S_hits","G_hits","n_sections","E_rate","S_rate","G_rate"
    ]).head(10)
)


Saved table → outputs\esg_panel.sqlite (table: firm_year_corpus)
SQLite setup → indexes created/verified
shape: (10, 12)
┌────────┬────────────┬──────┬──────────────────┬───┬────────────┬───────────┬──────────┬──────────┐
│ ticker ┆ cik        ┆ year ┆ gics_sector      ┆ … ┆ n_sections ┆ E_rate    ┆ S_rate   ┆ G_rate   │
│ ---    ┆ ---        ┆ ---  ┆ ---              ┆   ┆ ---        ┆ ---       ┆ ---      ┆ ---      │
│ str    ┆ str        ┆ i32  ┆ str              ┆   ┆ u32        ┆ f64       ┆ f64      ┆ f64      │
╞════════╪════════════╪══════╪══════════════════╪═══╪════════════╪═══════════╪══════════╪══════════╡
│ WM     ┆ 0000823768 ┆ 2020 ┆ Industrials      ┆ … ┆ 2          ┆ 17.463574 ┆ 3.471674 ┆ 2.998264 │
│ DIS    ┆ 0001744489 ┆ 2023 ┆ Communication    ┆ … ┆ 2          ┆ 0.972868  ┆ 6.972219 ┆ 1.51335  │
│        ┆            ┆      ┆ Services         ┆   ┆            ┆           ┆          ┆          │
│ TFC    ┆ 0000092230 ┆ 2023 ┆ Financials       ┆ … ┆ 2          ┆ 1.34

In [12]:
uv pip install sqlalchemy

Note: you may need to restart the kernel to use updated packages.


Using Python 3.13.7 environment at: c:\Users\ddddd\AppData\Local\Programs\Python\Python313
Resolved 3 packages in 1.13s
Prepared 2 packages in 1.25s
Installed 3 packages in 194ms
 + greenlet==3.3.1
 + sqlalchemy==2.0.46
 + typing-extensions==4.15.0


DatabaseError: Execution failed on sql 'SELECT * FROM firm_year_corpus': (sqlite3.OperationalError) no such table: firm_year_corpus
[SQL: SELECT * FROM firm_year_corpus]
(Background on this error at: https://sqlalche.me/e/20/e3q8)

In [16]:
uv pip install python-dotenv pymysql


Note: you may need to restart the kernel to use updated packages.


Using Python 3.13.7 environment at: c:\Users\ddddd\AppData\Local\Programs\Python\Python313
Resolved 2 packages in 814ms
Prepared 1 package in 135ms
Installed 2 packages in 147ms
 + pymysql==1.1.2
 + python-dotenv==1.2.1


In [18]:
#Search DB for information 


from pathlib import Path
import os
from urllib.parse import quote_plus

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

load_dotenv()

user = os.getenv("MYSQL_USER", "root")
pwd = os.getenv("MYSQL_PASSWORD", "")
host = os.getenv("MYSQL_HOST", "localhost")
port = os.getenv("MYSQL_PORT", "3306")
db   = os.getenv("MYSQL_DB", "fyp")

engine = create_engine(f"mysql+pymysql://{user}:{quote_plus(pwd)}@{host}:{port}/{db}")

TABLE_PREFIX = "firm_year_corpus_"

with engine.connect() as conn:
    # List tables
    tables = [r[0] for r in conn.execute(text("SHOW TABLES")).fetchall()]
    print(f"Database: {db}")
    print(f"Tables ({len(tables)}):")
    for t in sorted(tables):
        print("  -", t)

    # Find firm_year_corpus_XX tables and pick the latest
    candidates = []
    for t in tables:
        if t.startswith(TABLE_PREFIX):
            try:
                candidates.append((int(t.replace(TABLE_PREFIX, "")), t))
            except ValueError:
                pass

    if not candidates:
        raise RuntimeError(f"No tables found with prefix '{TABLE_PREFIX}' in database '{db}'")

    candidates.sort()
    latest_n, latest_table = candidates[-1]
    print(f"\nLatest corpus table: {latest_table} (version {latest_n:02d})")

    # Row count
    nrows = conn.execute(text(f"SELECT COUNT(*) FROM {latest_table}")).scalar()
    print(f"Rows: {nrows}")

    # Columns (schema)
    cols = conn.execute(text(f"SHOW COLUMNS FROM {latest_table}")).fetchall()
    print("\nSchema:")
    for field, coltype, null, key, default, extra in cols:
        print(f"  - {field}: {coltype} {'(PK)' if key == 'PRI' else ''}")

    # Sample rows (small)
    sample = pd.read_sql(text(f"SELECT ticker, cik, year, gics_sector, tokens, E_rate, S_rate, G_rate "
                              f"FROM {latest_table} ORDER BY year DESC LIMIT 10"), conn)
    print("\nSample (latest years):")
    print(sample)


Database: fyp
Tables (4):
  - firm_year_corpus_01
  - firm_year_corpus_02
  - firm_year_corpus_03
  - firm_year_corpus_04

Latest corpus table: firm_year_corpus_04 (version 04)
Rows: 4929

Schema:
  - ticker: text 
  - cik: text 
  - year: bigint 
  - gics_sector: text 
  - tokens: bigint 
  - E_hits: bigint 
  - S_hits: bigint 
  - G_hits: bigint 
  - n_sections: bigint 
  - E_rate: double 
  - S_rate: double 
  - G_rate: double 

Sample (latest years):
  ticker      cik  year             gics_sector  tokens    E_rate    S_rate  \
0    PSA  1393311  2025             Real Estate   12522  4.791567  8.864399   
1    CAH   721371  2025             Health Care    7497  2.534347  4.134987   
2   EPAM  1352010  2025  Information Technology   16637  0.901605  9.076156   
3   ZBRA   877212  2025  Information Technology   15827  1.769129  6.065584   
4    BLK  2012383  2025              Financials   31742  1.354672  7.277424   
5    STZ    16918  2025        Consumer Staples   18003  3.110593  

In [19]:
# Take the data and panel accordingly
import os
from urllib.parse import quote_plus

import pandas as pd
import polars as pl
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

load_dotenv()

user = os.getenv("MYSQL_USER", "root")
pwd = os.getenv("MYSQL_PASSWORD", "")
host = os.getenv("MYSQL_HOST", "localhost")
port = os.getenv("MYSQL_PORT", "3306")
db   = os.getenv("MYSQL_DB", "fyp")

engine = create_engine(f"mysql+pymysql://{user}:{quote_plus(pwd)}@{host}:{port}/{db}")

TABLE_PREFIX = "firm_year_corpus_"

# --------------------
# Find latest corpus table
# --------------------
with engine.connect() as conn:
    tables = [r[0] for r in conn.execute(text("SHOW TABLES")).fetchall()]

candidates = []
for t in tables:
    if t.startswith(TABLE_PREFIX):
        try:
            candidates.append((int(t.replace(TABLE_PREFIX, "")), t))
        except ValueError:
            pass

assert candidates, f"No tables found with prefix '{TABLE_PREFIX}' in database '{db}'"
candidates.sort()
src_n, TABLE_IN = candidates[-1]
TABLE_OUT = f"panel_firm_year_corpus_{src_n:02d}"

print(f"Source → {db}.{TABLE_IN}")
print(f"Will write → {db}.{TABLE_OUT}")

# --------------------
# Read to Polars
# --------------------
with engine.connect() as conn:
    df_pd = pd.read_sql(text(f"SELECT * FROM {TABLE_IN}"), conn)

firm_year_corpus = pl.from_pandas(df_pd)

# --------------------
# Types + required cols
# --------------------
required = {"ticker", "cik", "year", "gics_sector", "tokens", "E_rate", "S_rate", "G_rate"}
missing = required - set(firm_year_corpus.columns)
assert not missing, f"Missing columns in {TABLE_IN}: {missing}"

firm_year_corpus = firm_year_corpus.with_columns(
    pl.col("ticker").cast(pl.Utf8),
    pl.col("cik").cast(pl.Utf8),
    pl.col("gics_sector").cast(pl.Utf8),
    pl.col("year").cast(pl.Int32),
    pl.col("tokens").cast(pl.Int64),
    pl.col("E_rate").cast(pl.Float64),
    pl.col("S_rate").cast(pl.Float64),
    pl.col("G_rate").cast(pl.Float64),
)

# --------------------
# Panel features
# --------------------
min_year = firm_year_corpus.select(pl.col("year").min()).item()

panel_tbl = firm_year_corpus.with_columns(
    (pl.col("year") >= 2015).cast(pl.Int8).alias("post_2015"),
    (pl.col("year") >= 2020).cast(pl.Int8).alias("post_2020"),
    (pl.col("year") >= 2021).cast(pl.Int8).alias("post_2021"),
    (pl.col("year") - pl.lit(min_year)).cast(pl.Int32).alias("t"),
)

# --------------------
# Write back to MySQL (fail if exists; version is tied to source XX)
# --------------------
panel_tbl.write_database(
    table_name=TABLE_OUT,
    connection=engine,
    if_table_exists="fail",
)

print(f"Wrote panel table → {db}.{TABLE_OUT}")

# --------------------
# Indexes (ticker/cik are TEXT-ish in MySQL => prefix indexes)
# --------------------
def try_exec(conn, stmt: str) -> None:
    try:
        conn.execute(text(stmt))
        print(f"Index OK → {stmt}")
    except Exception as e:
        print(f"Index skipped → {str(e).splitlines()[0]}")

with engine.begin() as conn:
    try_exec(conn, f"CREATE INDEX idx_{TABLE_OUT}_ticker_year ON {TABLE_OUT} (ticker(16), year)")
    try_exec(conn, f"CREATE INDEX idx_{TABLE_OUT}_cik_year ON {TABLE_OUT} (cik(16), year)")
    try_exec(conn, f"CREATE INDEX idx_{TABLE_OUT}_sector_year ON {TABLE_OUT} (gics_sector(32), year)")

# Minimal preview to console
print(panel_tbl.select(["ticker","year","tokens","E_rate","post_2015","post_2021"]).head(10))


Source → fyp.firm_year_corpus_04
Will write → fyp.panel_firm_year_corpus_04
Wrote panel table → fyp.panel_firm_year_corpus_04
Index OK → CREATE INDEX idx_panel_firm_year_corpus_04_ticker_year ON panel_firm_year_corpus_04 (ticker(16), year)
Index OK → CREATE INDEX idx_panel_firm_year_corpus_04_cik_year ON panel_firm_year_corpus_04 (cik(16), year)
Index OK → CREATE INDEX idx_panel_firm_year_corpus_04_sector_year ON panel_firm_year_corpus_04 (gics_sector(32), year)
shape: (10, 6)
┌────────┬──────┬────────┬──────────┬───────────┬───────────┐
│ ticker ┆ year ┆ tokens ┆ E_rate   ┆ post_2015 ┆ post_2021 │
│ ---    ┆ ---  ┆ ---    ┆ ---      ┆ ---       ┆ ---       │
│ str    ┆ i32  ┆ i64    ┆ f64      ┆ i8        ┆ i8        │
╞════════╪══════╪════════╪══════════╪═══════════╪═══════════╡
│ ADI    ┆ 2024 ┆ 17662  ┆ 3.623599 ┆ 1         ┆ 1         │
│ IFF    ┆ 2019 ┆ 17208  ┆ 2.440725 ┆ 1         ┆ 0         │
│ EPAM   ┆ 2017 ┆ 15313  ┆ 0.065304 ┆ 1         ┆ 0         │
│ F      ┆ 2024 ┆ 2195

# Pull and save as PNGs
# This is test output to enable us to better understand our breakpoint data

In [21]:
from pathlib import Path
import os
from urllib.parse import quote_plus

import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

# --------------------
# Output (PNG) helpers: overwrite deterministically
# --------------------
OUT_DIR = Path("outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

def save_png_overwrite(name: str, dpi: int = 200) -> Path:
    path = OUT_DIR / f"{name}.png"
    if path.exists():
        path.unlink()  # delete previous run artefact
    plt.savefig(path, dpi=dpi, bbox_inches="tight")
    plt.close()
    print(f"Saved figure → {path}")
    return path

# --------------------
# MySQL connect (from .env)
# --------------------
load_dotenv()

user = os.getenv("MYSQL_USER", "root")
pwd = os.getenv("MYSQL_PASSWORD", "")
host = os.getenv("MYSQL_HOST", "localhost")
port = os.getenv("MYSQL_PORT", "3306")
db   = os.getenv("MYSQL_DB", "fyp")

engine = create_engine(f"mysql+pymysql://{user}:{quote_plus(pwd)}@{host}:{port}/{db}")

# --------------------
# Pick latest panel table
# --------------------
PANEL_PREFIX = "panel_firm_year_corpus_"

with engine.connect() as conn:
    tables = [r[0] for r in conn.execute(text("SHOW TABLES")).fetchall()]

candidates = []
for t in tables:
    if t.startswith(PANEL_PREFIX):
        try:
            candidates.append((int(t.replace(PANEL_PREFIX, "")), t))
        except ValueError:
            pass

assert candidates, f"No panel tables found with prefix '{PANEL_PREFIX}' in database '{db}'"
candidates.sort()
src_n, PANEL_TABLE = candidates[-1]

print(f"Using panel table → {db}.{PANEL_TABLE}")

# --------------------
# Identify IT sector label robustly
# --------------------
with engine.connect() as conn:
    it_rows = conn.execute(text(f"""
        SELECT DISTINCT gics_sector
        FROM {PANEL_TABLE}
        WHERE gics_sector IS NOT NULL
          AND LOWER(gics_sector) LIKE '%information%'
          AND LOWER(gics_sector) LIKE '%technology%'
    """)).fetchall()

assert it_rows, "No IT sector label found. Inspect DISTINCT gics_sector in the panel table."
IT_SECTOR = it_rows[0][0]
print(f"IT sector label → {IT_SECTOR}")

# --------------------
# Token-weighted rates by year (All sectors vs IT)
# rate = 1000 * SUM(hits) / SUM(tokens)
# --------------------
all_year = pd.read_sql(text(f"""
    SELECT
        year,
        (SUM(E_hits) * 1000.0 / NULLIF(SUM(tokens), 0)) AS E_w,
        (SUM(S_hits) * 1000.0 / NULLIF(SUM(tokens), 0)) AS S_w,
        (SUM(G_hits) * 1000.0 / NULLIF(SUM(tokens), 0)) AS G_w,
        COUNT(DISTINCT cik) AS n_firms
    FROM {PANEL_TABLE}
    GROUP BY year
    ORDER BY year
"""), engine)

it_year = pd.read_sql(text(f"""
    SELECT
        year,
        (SUM(E_hits) * 1000.0 / NULLIF(SUM(tokens), 0)) AS E_w,
        (SUM(S_hits) * 1000.0 / NULLIF(SUM(tokens), 0)) AS S_w,
        (SUM(G_hits) * 1000.0 / NULLIF(SUM(tokens), 0)) AS G_w,
        COUNT(DISTINCT cik) AS n_firms
    FROM {PANEL_TABLE}
    WHERE gics_sector = :it
    GROUP BY year
    ORDER BY year
"""), engine, params={"it": IT_SECTOR})

# --------------------
# Plot: Environmental (token-weighted)
# --------------------
plt.figure()
plt.plot(all_year["year"], all_year["E_w"], label="All sectors")
plt.plot(it_year["year"], it_year["E_w"], label="Information Technology")
plt.xlabel("Year")
plt.ylabel("E_rate (token-weighted; hits per 1,000 tokens)")
plt.title("Environmental disclosure intensity (token-weighted): IT vs All sectors")
plt.legend()
save_png_overwrite(f"it_vs_all_E_token_weighted_{src_n:02d}")

# --------------------
# Plot: Social (token-weighted)
# --------------------
plt.figure()
plt.plot(all_year["year"], all_year["S_w"], label="All sectors")
plt.plot(it_year["year"], it_year["S_w"], label="Information Technology")
plt.xlabel("Year")
plt.ylabel("S_rate (token-weighted; hits per 1,000 tokens)")
plt.title("Social disclosure intensity (token-weighted): IT vs All sectors")
plt.legend()
save_png_overwrite(f"it_vs_all_S_token_weighted_{src_n:02d}")

# --------------------
# Plot: Governance (token-weighted)
# --------------------
plt.figure()
plt.plot(all_year["year"], all_year["G_w"], label="All sectors")
plt.plot(it_year["year"], it_year["G_w"], label="Information Technology")
plt.xlabel("Year")
plt.ylabel("G_rate (token-weighted; hits per 1,000 tokens)")
plt.title("Governance disclosure intensity (token-weighted): IT vs All sectors")
plt.legend()
save_png_overwrite(f"it_vs_all_G_token_weighted_{src_n:02d}")

# --------------------
# Plot: IT coverage (firm count)
# --------------------
plt.figure()
plt.plot(it_year["year"], it_year["n_firms"], label="IT firms in sample")
plt.xlabel("Year")
plt.ylabel("Number of firms")
plt.title("Information Technology sample coverage by year")
plt.legend()
save_png_overwrite(f"it_firm_count_{src_n:02d}")


Using panel table → fyp.panel_firm_year_corpus_04
IT sector label → Information Technology
Saved figure → outputs\it_vs_all_E_token_weighted_04.png
Saved figure → outputs\it_vs_all_S_token_weighted_04.png
Saved figure → outputs\it_vs_all_G_token_weighted_04.png
Saved figure → outputs\it_firm_count_04.png


WindowsPath('outputs/it_firm_count_04.png')

# Interesting observation
# Using IT as our control variable, we find that while there is high positive correlation between IT and other sectors of the S&P 500 in relation to an inrease in Social discourse. Governance discourse is less correlated but still rises significantly within 2020 - 2021 for IT and other sectors, Environmental discourse actually diverges past this period

